# Stage 08 closeout on SageMaker

This notebook runs the final structural fast-track validation and packages the final closeout bundle.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path.cwd()
print('Working directory:', ROOT)
assert (ROOT / 'pyproject.toml').exists(), 'Run this notebook from the repo root in SageMaker JupyterLab.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Environment ready')


In [ ]:
TOP3_OUT = ROOT / 'results' / 'stage08' / 'structural_fasttrack_top3'
TOP5_OUT = ROOT / 'results' / 'stage08' / 'structural_fasttrack_top5'
FINAL_OUT = ROOT / 'results' / 'final_closeout'
for p in [TOP3_OUT, TOP5_OUT, FINAL_OUT]:
    p.mkdir(parents=True, exist_ok=True)
print(TOP3_OUT, TOP5_OUT, FINAL_OUT, sep='
')


## Run Stage 08 on validated top-3


In [ ]:
cmd = [
    sys.executable, 'scripts/08a_structural_fasttrack_validation.py',
    '--validated_csv', 'results/stage07/final_validation/validated_top3.csv',
    '--ranked_csv', 'results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv',
    '--context_json', 'results/stage07/context/stage07_context.base.json',
    '--out_dir', str(TOP3_OUT),
    '--top_k', '3',
    '--device', 'cuda',
    '--chunk_size', '128',
    '--num_recycles', '1',
    '--resume',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## Run Stage 08 on validated top-5


In [ ]:
cmd = [
    sys.executable, 'scripts/08a_structural_fasttrack_validation.py',
    '--validated_csv', 'results/stage07/final_validation/validated_top5.csv',
    '--ranked_csv', 'results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv',
    '--context_json', 'results/stage07/context/stage07_context.base.json',
    '--out_dir', str(TOP5_OUT),
    '--top_k', '5',
    '--device', 'cuda',
    '--chunk_size', '128',
    '--num_recycles', '1',
    '--resume',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## Package the final closeout bundle


In [ ]:
cmd = [
    sys.executable, 'scripts/08b_make_final_closeout.py',
    '--ranked_csv', 'results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv',
    '--validated_top3_csv', 'results/stage07/final_validation/validated_top3.csv',
    '--validated_top5_csv', 'results/stage07/final_validation/validated_top5.csv',
    '--stage08_top3_csv', str(TOP3_OUT / 'stage08_structural_fasttrack_summary.csv'),
    '--stage08_top5_csv', str(TOP5_OUT / 'stage08_structural_fasttrack_summary.csv'),
    '--context_json', 'results/stage07/context/stage07_context.base.json',
    '--out_dir', str(FINAL_OUT),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## Inspect the final outputs


In [ ]:
import pandas as pd
top3 = pd.read_csv(TOP3_OUT / 'stage08_structural_fasttrack_summary.csv')
top5 = pd.read_csv(TOP5_OUT / 'stage08_structural_fasttrack_summary.csv')
final_df = pd.read_csv(FINAL_OUT / 'final_candidate_table.csv')
display(top3)
display(top5)
display(final_df)


## Optional: zip the final deliverables in SageMaker


In [ ]:
zip_path = ROOT / 'results' / 'final_closeout_bundle.zip'
subprocess.run(['bash', '-lc', f"cd {ROOT} && zip -r {zip_path} results/final_closeout results/stage08/structural_fasttrack_top3 results/stage08/structural_fasttrack_top5"], check=True)
print('Created:', zip_path)
